In [2]:
using LinearAlgebra

# Matrix multiplication benchmark
println("Testing matrix multiplication speed...")
A = rand(1000, 1000)
B = rand(1000, 1000)
@time A * B

# Expected times:
# Modern desktop: 10-30 ms
# Decent laptop: 30-100 ms
# Older laptop: 100-300 ms

Testing matrix multiplication speed...
  1.475948 seconds (4.48 M allocations: 229.753 MiB, 16.24% gc time, 97.47% compilation time)


1000×1000 Matrix{Float64}:
 252.606  247.05   246.413  257.636  …  253.957  258.115  254.016  268.125
 262.532  254.031  260.698  270.257     263.579  274.115  265.25   274.4
 244.815  244.452  247.373  248.222     245.783  256.089  242.627  260.148
 247.28   243.137  252.133  254.765     250.671  256.392  251.268  266.713
 248.659  238.942  249.203  254.875     246.695  250.248  247.072  256.772
 251.2    246.36   250.65   258.58   …  256.43   257.715  251.165  262.574
 245.09   239.497  241.467  250.812     247.075  247.406  245.937  256.911
 258.701  251.55   262.818  265.928     261.462  267.713  263.005  275.939
 247.588  241.526  250.731  255.589     253.508  253.573  248.671  263.487
 257.112  242.773  253.866  259.64      258.41   261.438  252.298  263.734
 251.128  242.325  253.951  255.271  …  252.038  254.919  245.007  259.585
 248.317  242.118  247.339  254.913     249.902  252.329  247.401  260.441
 249.558  243.955  250.684  254.773     251.15   254.39   250.922  264.77
 

In [3]:
# Simple ODE benchmark
using DifferentialEquations

function lorenz!(du, u, p, t)
    du[1] = 10.0 * (u[2] - u[1])
    du[2] = u[1] * (28.0 - u[3]) - u[2]
    du[3] = u[1] * u[2] - (8/3) * u[3]
end

u0 = [1.0, 0.0, 0.0]
tspan = (0.0, 100.0)
prob_bench = ODEProblem(lorenz!, u0, tspan)

println("\nTesting ODE solver speed (Lorenz system)...")
@time sol_bench = solve(prob_bench, Rodas5())

# Expected times:
# Modern desktop: 0.01-0.05 seconds
# Decent laptop: 0.05-0.2 seconds
# Older laptop: 0.2-1.0 seconds


Testing ODE solver speed (Lorenz system)...
  4.238027 seconds (4.45 M allocations: 234.037 MiB, 4.43% gc time, 99.78% compilation time)


retcode: Success
Interpolation: specialized 4rd order "free" stiffness-aware interpolation
t: 1422-element Vector{Float64}:
   0.0
   3.5678604836301404e-5
   0.0003924646531993154
   0.003219523954374159
   0.010130792498516391
   0.019221699189546783
   0.03131202799578121
   0.04678203688141252
   0.06638539030431816
   0.09013871628784907
   0.11752372077027054
   0.14771260886765322
   0.18025028950642913
   ⋮
  99.31722678454234
  99.36622716669002
  99.4152275488377
  99.46876301934203
  99.52229848984636
  99.58672056846802
  99.6610901903651
  99.73545981226218
  99.80982943415925
  99.88419905605633
  99.96284280554244
 100.0
u: 1422-element Vector{Vector{Float64}}:
 [1.0, 0.0, 0.0]
 [0.9996434557625105, 0.0009988049817849076, 1.7814347887992186e-8]
 [0.996104549742581, 0.010965399721242447, 2.146955365852547e-6]
 [0.9697366941267918, 0.0886102562861196, 0.00014010971935064003]
 [0.9170749617292998, 0.269664830773501, 0.0012956125778399218]
 [0.8707747383226055, 0.49318966291

In [4]:
# Check Julia threads and system info
println("\nSystem information:")
println("Julia version: ", VERSION)
println("Number of threads: ", Threads.nthreads())
println("CPU: ", Sys.cpu_info()[1].model)
println("Total RAM: ", round(Sys.total_memory() / 2^30, digits=1), " GB")

# Check if you're using optimized BLAS
using LinearAlgebra
println("BLAS config: ", BLAS.get_config())


System information:
1.11.5version: 
Number of threads: 1
CPU: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
15.8 GBAM: 
LBTConfig([ILP64] libopenblas64_.dll)


In [5]:
# Simplified reaction network benchmark
function simple_reactions!(du, u, p, t)
    k = [1.0, 0.1, 0.5]  # Rate constants
    
    # A + B -> C (k1)
    rate1 = k[1] * u[1] * u[2]
    # C -> A + B (k2)
    rate2 = k[2] * u[3]
    # C -> D (k3)
    rate3 = k[3] * u[3]
    
    du[1] = -rate1 + rate2  # A
    du[2] = -rate1 + rate2  # B
    du[3] = rate1 - rate2 - rate3  # C
    du[4] = rate3  # D
end

# 10 nodes, 4 species each = 40 ODEs
u0_bench = repeat([1.0, 1.0, 0.0, 0.0], 10)
prob_chem = ODEProblem(simple_reactions!, u0_bench, (0.0, 10.0))

println("\nTesting 40-ODE chemical system...")
@time sol_chem = solve(prob_chem, Rodas5(), abstol=1e-6, reltol=1e-4)

# Expected times:
# Modern desktop: 0.05-0.2 seconds
# Decent laptop: 0.2-0.5 seconds
# Older laptop: 0.5-2.0 seconds


Testing 40-ODE chemical system...
  0.793513 seconds (856.06 k allocations: 49.244 MiB, 6.87% gc time, 89.65% compilation time)


retcode: Success
Interpolation: specialized 4rd order "free" stiffness-aware interpolation
t: 19-element Vector{Float64}:
  0.0
  0.029898066669076923
  0.08971480049677882
  0.17441658197124613
  0.2869952100814324
  0.4347034678033449
  0.6232228328065331
  0.8617031643969937
  1.1607319805603071
  1.5356666008081208
  2.0073245845821193
  2.6035912780436763
  3.355244084807384
  4.2849015891270525
  5.349575949795799
  6.524855670201755
  7.8164327508349
  9.2252342669087
 10.0
u: 19-element Vector{Vector{Float64}}:
 [1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0  …  0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [0.9710125974248383, 0.9710125974248383, 0.028769593299751928, 0.00021780927540978999, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0  …  0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [0.9180239751571985, 0.9180239751571985, 0.08011049913747258, 0.0018655257053288392, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0  …  0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
 [0.8526740746476047, 0.852674